# Fase 6 - Despliegue

La aplicación Streamlit está en `Streamlit app/app.py` y consume los seis pipelines persistidos en `Modelos/`.

## 6.1 Componentes entregados
| Componente | Ubicación |
|---|---|
| ETL reutilizable | `src/etl.py` |
| EDA reutilizable | `src/eda.py` |
| Modelado y evaluación | `src/modeling.py` |
| Reentrenamiento de modelos | `src/train_models.py` |
| Modelos y métricas | `Modelos/` |
| Aplicación | `Streamlit app/app.py` |

Módulos de la app (según el encargo):
1. **Selección de variables:** permite escoger los predictores; con la selección por defecto usa los modelos persistidos y con otra selección entrena una configuración exploratoria en caché sin sobrescribir artefactos.
2. **Consultas en tiempo real:** formulario que devuelve la predicción y probabilidad de los modelos de ambas iteraciones.
3. **Evaluación de generalización:** ejecuta los modelos sobre el test ciego.
4. **Dashboard de comparación:** tabla de métricas, gráficos de barras, curvas ROC y matrices de confusión por iteración.

## 6.2 Verificación de los artefactos que usa la aplicación

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve()
while not (ROOT / "Data" / "bank-full.csv").is_file() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
DATA_PATH = ROOT / "Data" / "bank-full.csv"
MODELS_DIR = ROOT / "Modelos"

import json
import pandas as pd

meta = json.loads((MODELS_DIR / "metadata.json").read_text(encoding="utf-8"))
print("Modelo recomendado:", meta["selected_model"])
print("Predictores:", meta["selected_features"])
pd.DataFrame({"archivo": meta["models"].values(),
              "tamano_KB": [round((MODELS_DIR / f).stat().st_size / 1024, 1) for f in meta["models"].values()]},
             index=meta["models"].keys())

Modelo recomendado: Iteracion 2 | Arbol de decision
Predictores: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'pdays', 'previous', 'poutcome']


,archivo,tamano_KB
Iteracion 1 | Regresion logistica,iteracion_1_regresion_logistica.joblib,5.9
Iteracion 2 | Regresion logistica,iteracion_2_regresion_logistica.joblib,6.5
Iteracion 1 | Arbol de decision,iteracion_1_arbol_de_decision.joblib,701.9
Iteracion 2 | Arbol de decision,iteracion_2_arbol_de_decision.joblib,15.4
Iteracion 1 | Random Forest,iteracion_1_random_forest.joblib,5622.0
Iteracion 2 | Random Forest,iteracion_2_random_forest.joblib,7389.0


## 6.3 Ejecución local
```powershell
.venv\Scripts\python.exe -m pip install -r "Streamlit app\requirements.txt"
.venv\Scripts\python.exe -m streamlit run "Streamlit app\app.py"
```
Para regenerar los modelos: `.venv\Scripts\python.exe -m src.train_models`.

## 6.4 Controles antes de producción
- Monitorear deriva de datos y recalibrar umbrales periódicamente.
- Validar con un piloto controlado antes de usar las predicciones en decisiones comerciales.
- Los datos personales y el cumplimiento normativo deben revisarse antes de usar la herramienta con clientes reales.

## 6.5 Conclusión general del proyecto (metodología CRISP-DM)
Se recorrieron las seis fases: negocio, datos, preparación, modelado con **2 iteraciones × 3 algoritmos**, evaluación con test ciego y despliegue en Streamlit. La Iteración 2 corrigió la limitación principal de la base (Recall muy bajo por el desbalance) mediante selección de características, pesos de clase y ajuste de hiperparámetros. El modelo recomendado es el Árbol de Decisión de la Iteración 2 por su equilibrio y simplicidad; el Random Forest y la regresión logística balanceados cumplen los umbrales de Recall y ROC-AUC. El desempeño es moderado, por lo que el sistema debe usarse como apoyo a la priorización y validarse en un piloto.